In [1]:
import tensorflow as tf


2026-03-22 16:03:22.763250: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from tensorflow import keras

In [7]:
from keras.utils import image_dataset_from_directory

In [36]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
# python standard library imports
from typing import Self, Any
from pathlib import Path
from keras.utils import image_dataset_from_directory
import numpy as np
from sklearn.model_selection import train_test_split

In [39]:
from sklearn.utils.class_weight import compute_class_weight

In [28]:
dataset = Path("./wikiart")
image_size = (224, 224)
batch_size = 32
seed = 42
val_split = 0.2
test_split = 0.1

In [ ]:
images = list(dataset.glob("*/*"))
labels = [p.parent.name for p in images]   # list of all image labels
classes = sorted(list(set(labels))) # set of classes
label_to_index = {c: i for i, c in enumerate(classes)} # dict label:index
labels_encoded = np.array([label_to_index[l] for l in labels]) # all labels encoded (1-23)
num_classes = len(classes)


In [ ]:
print("No. of Classes:", num_classes) # certo


No. of Classes: 23


In [37]:
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    images, labels_encoded, test_size=test_split, stratify=labels_encoded, random_state=seed)


train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=val_split, stratify=train_val_labels, random_state=seed)

In [ ]:
class_weights = compute_class_weight(class_weight='balanced',classes=np.unique(train_labels), y=train_labels)
class_weights_dict = dict(enumerate(class_weights))


In [41]:
class_weights_dict

{0: 1.001355437389219,
 1: 1.304891304347826,
 2: 0.934150374477191,
 3: 1.5074556584523622,
 4: 0.6204535176691001,
 5: 1.3557312252964426,
 6: 1.491304347826087,
 7: 1.0988558352402746,
 8: 1.5351662404092072,
 9: 1.434932018526819,
 10: 1.5937603717225357,
 11: 1.0571271326362135,
 12: 1.0817751745888713,
 13: 1.439880059970015,
 14: 0.45536010620643874,
 15: 1.084584980237154,
 16: 1.4300178677784396,
 17: 0.5956707808720462,
 18: 0.8999250374812594,
 19: 1.5998667333000167,
 20: 1.065217391304348,
 21: 1.7254761049227452,
 22: 0.43861892583120204}

In [ ]:

def create_dataset(images, labels, num_classes, batch_size=32, augment=False):
    
    "Converts each image into a tensor with color channels and one-hot encodes the labels."


    images_x = tf.data.Dataset.from_tensor_slices([str(i) for i in images])
    labels_y = tf.data.Dataset.from_tensor_slices(labels)
    ds = tf.data.Dataset.zip((images_x, labels_y)) # zip of image paths + labels


    def load_image(image, label):
        img = tf.io.read_file(image)
        img = tf.image.decode_jpeg(img, channels=3) # converts to tensor w/ color channels
        img = tf.image.resize(img, (224, 224)) # resizes to 224:224 px 
        img = tf.cast(img, tf.float32) / 255.0  # normalizes px for easier computation
        
        
        label = tf.one_hot(label, depth=num_classes) # one hot labels
        
        return img, label # tuple: tensor + label

    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE) # applies load_image to every pair in the ds zip 
    # every pair in ds is now a tensor + label

    # augment only for training set (fix this for class imbalance n ta a fazer nada )
    if augment:
        data_aug = tf.keras.Sequential([
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomRotation(0.1),
            tf.keras.layers.RandomZoom(0.1),
        ])
        ds = ds.map(lambda x, y: (data_aug(x), y), num_parallel_calls=tf.data.AUTOTUNE) # augmentation applied to ds zip 

    
    ds = ds.shuffle(buffer_size=len(images_x)) # shuffles
    ds = ds.batch(batch_size) # divides data into batches 
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [43]:
train_ds = create_dataset(train_paths, train_labels, num_classes, augment=True)
val_ds   = create_dataset(val_paths, val_labels, num_classes)
test_ds  = create_dataset(test_paths, test_labels, num_classes)

In [48]:
train_ds

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 23), dtype=tf.float32, name=None))>

A shape é (batch_size, 224, 224, 3) (tensor), (batch_size, 23) (labels)

In [50]:

model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy', # because we have one hot encoded labels
    metrics=['accuracy']
)

/Users/beatriz/Desktop/MESTRADO/Deep Learning/deep-learning-env/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [51]:

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weights_dict # use class weights because of the imbalance
)


test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.4f}")

Epoch 1/5


2026-03-22 19:35:22.765721: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 1182 of 9604
2026-03-22 19:35:42.761537: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 3163 of 9604
2026-03-22 19:36:02.763396: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 5332 of 9604
2026-03-22 19:36:22.761006: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 6899 of 9604
2026-03-22 19:36:32.783659: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 7837 of 9604
2026-03-22 19:36:52.762747: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a w

301/301 ━━━━━━━━━━━━━━━━━━━━ 699s 2s/step - accuracy: 0.1152 - loss: 3.0299 - val_accuracy: 0.1669 - val_loss: 3.1502
Epoch 2/5


2026-03-22 19:46:58.363162: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 1690 of 9604
2026-03-22 19:47:08.366829: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 3392 of 9604
2026-03-22 19:47:28.365918: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 6538 of 9604
2026-03-22 19:47:46.379636: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


301/301 ━━━━━━━━━━━━━━━━━━━━ 646s 2s/step - accuracy: 0.2186 - loss: 2.6496 - val_accuracy: 0.2506 - val_loss: 2.5542
Epoch 3/5


...

301/301 ━━━━━━━━━━━━━━━━━━━━ 726s 2s/step - accuracy: 0.2658 - loss: 2.4378 - val_accuracy: 0.2885 - val_loss: 2.4239
Epoch 4/5


2026-03-22 20:09:50.377661: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 1441 of 9604
2026-03-22 20:10:00.388305: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 2747 of 9604
2026-03-22 20:10:10.393297: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 4143 of 9604
2026-03-22 20:10:30.395605: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 7007 of 9604
2026-03-22 20:10:48.692224: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


301/301 ━━━━━━━━━━━━━━━━━━━━ 660s 2s/step - accuracy: 0.3085 - loss: 2.2836 - val_accuracy: 0.3006 - val_loss: 2.3706
Epoch 5/5


2026-03-22 20:20:50.037993: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 1509 of 9604
2026-03-22 20:21:10.037747: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 4464 of 9604
2026-03-22 20:21:30.042148: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 7065 of 9604
2026-03-22 20:21:47.017832: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


301/301 ━━━━━━━━━━━━━━━━━━━━ 699s 2s/step - accuracy: 0.3341 - loss: 2.1833 - val_accuracy: 0.3093 - val_loss: 2.4743
42/42 ━━━━━━━━━━━━━━━━━━━━ 25s 535ms/step - accuracy: 0.3088 - loss: 2.4575
Test Accuracy: 0.3088


DATASET IS NOT BALANCED! To do: oversample minority classes with augmentation or find another way.